In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE

print("Loading Hadoop data...")
df = pd.read_csv("/content/drive/MyDrive/hadoop_openstack/hadoop_logs_parsed.csv")
print(f"Shape: {df.shape}")
print(f"\nLabel Distribution:")
print(df['label'].value_counts())

# Features
le_app = LabelEncoder()
le_container = LabelEncoder()
df['app_id_enc'] = le_app.fit_transform(df['app_id'])
df['container_id_enc'] = le_container.fit_transform(df['container_id'])

features = ['cluster_id', 'app_id_enc', 'container_id_enc']
X = df[features].values
y = df['label'].values

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# SMOTE
print("\nSMOTE apply ho raha hai...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"Normal: {(y_train_sm==0).sum():,}")
print(f"Anomaly: {(y_train_sm==1).sum():,}")
print(f"X_test: {X_test.shape}")
print("\nSab ready hai! ✅")

Mounted at /content/drive
Loading Hadoop data...
Shape: (393433, 7)

Label Distribution:
label
1    368148
0     25285
Name: count, dtype: int64

SMOTE apply ho raha hai...

After SMOTE:
Normal: 294,518
Anomaly: 294,518
X_test: (78687, 3)

Sab ready hai! ✅


In [3]:
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import SGDOneClassSVM
from sklearn.preprocessing import StandardScaler
import numpy as np

results = {}

# ============================================
# Model 1: Isolation Forest
# ============================================
print("=" * 45)
print("1. Isolation Forest training...")
iso = IsolationForest(n_estimators=100,
                      contamination=0.5,  # max allowed 0.5
                      random_state=42, n_jobs=-1)
iso.fit(X_train)
y_pred_iso = np.where(iso.predict(X_test) == -1, 1, 0)
scores_iso = -iso.decision_function(X_test)
roc_iso = roc_auc_score(y_test, scores_iso)
results['Isolation Forest'] = {
    'roc_auc': roc_iso,
    'report': classification_report(y_test, y_pred_iso,
                target_names=['Normal', 'Anomaly'])
}
print(f"ROC-AUC: {roc_iso:.4f} ✅")

# ============================================
# Model 2: Random Forest
# ============================================
print("2. Random Forest training...")
rf = RandomForestClassifier(n_estimators=100, random_state=42,
                             n_jobs=-1, class_weight='balanced')
rf.fit(X_train_sm, y_train_sm)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]
roc_rf = roc_auc_score(y_test, y_prob_rf)
results['Random Forest'] = {
    'roc_auc': roc_rf,
    'report': classification_report(y_test, y_pred_rf,
                target_names=['Normal', 'Anomaly'])
}
print(f"ROC-AUC: {roc_rf:.4f} ✅")

# ============================================
# Model 3: SGD One-Class SVM
# ============================================
print("3. SGD One-Class SVM training...")
scaler = StandardScaler()
X_train_normal = X_train[y_train == 0]
X_train_scaled = scaler.fit_transform(X_train_normal)
X_test_scaled = scaler.transform(X_test)

oc_svm = SGDOneClassSVM(nu=0.5, random_state=42)
oc_svm.fit(X_train_scaled)
y_pred_svm = np.where(oc_svm.predict(X_test_scaled) == -1, 1, 0)
scores_svm = -oc_svm.decision_function(X_test_scaled)
roc_svm = roc_auc_score(y_test, scores_svm)
results['One-Class SVM'] = {
    'roc_auc': roc_svm,
    'report': classification_report(y_test, y_pred_svm,
                target_names=['Normal', 'Anomaly'])
}
print(f"ROC-AUC: {roc_svm:.4f} ✅")

# ============================================
# Final Comparison
# ============================================
print("\n" + "=" * 45)
print("HADOOP — TRADITIONAL ML RESULTS")
print("=" * 45)
for model, res in results.items():
    print(f"\n{model} — ROC-AUC: {res['roc_auc']:.4f}")
    print(res['report'])

1. Isolation Forest training...
ROC-AUC: 0.2092 ✅
2. Random Forest training...
ROC-AUC: 1.0000 ✅
3. SGD One-Class SVM training...
ROC-AUC: 0.2383 ✅

HADOOP — TRADITIONAL ML RESULTS

Isolation Forest — ROC-AUC: 0.2092
              precision    recall  f1-score   support

      Normal       0.00      0.01      0.00      5057
     Anomaly       0.87      0.46      0.60     73630

    accuracy                           0.43     78687
   macro avg       0.44      0.24      0.30     78687
weighted avg       0.82      0.43      0.57     78687


Random Forest — ROC-AUC: 1.0000
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      5057
     Anomaly       1.00      1.00      1.00     73630

    accuracy                           1.00     78687
   macro avg       1.00      1.00      1.00     78687
weighted avg       1.00      1.00      1.00     78687


One-Class SVM — ROC-AUC: 0.2383
              precision    recall  f1-score   support

      No

In [4]:
import json, os

save_dir = "/content/drive/MyDrive/hadoop_openstack"

hadoop_trad_results = {
    "Random_Forest":    {"ROC_AUC": 1.0000, "Recall": 1.00, "F1": 1.00},
    "Isolation_Forest": {"ROC_AUC": 0.2092, "Recall": 0.46, "F1": 0.60},
    "OneClass_SVM":     {"ROC_AUC": 0.2383, "Recall": 0.20, "F1": 0.32}
}

with open(f"{save_dir}/hadoop_traditional_ml_results.json", "w") as f:
    json.dump(hadoop_trad_results, f, indent=4)

print("Saved! ✅")
print(json.dumps(hadoop_trad_results, indent=4))

Saved! ✅
{
    "Random_Forest": {
        "ROC_AUC": 1.0,
        "Recall": 1.0,
        "F1": 1.0
    },
    "Isolation_Forest": {
        "ROC_AUC": 0.2092,
        "Recall": 0.46,
        "F1": 0.6
    },
    "OneClass_SVM": {
        "ROC_AUC": 0.2383,
        "Recall": 0.2,
        "F1": 0.32
    }
}
